### 앙상블(Ensemble)
- 여러개의 분류 모델을 조합해서 더 나은 성능을 내느 방법
- Decision Tree 모델을 증가시켜 나온 랜덤포레스트가 대표적임

##### 랜덤포레스트
- 부트스트랩 샘플을 사용 : 부트 스트랩 샘플은 중복을 허용하는 샘플링 방법
- 결정트리의 과대적합 방지용
- 전체 갯수의 제곱근이 부트샘플링할 갯수

In [19]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [2]:
wine = pd.read_csv("../Data/wine.csv")
wine.head()

,alcohol,sugar,pH,class
0,9.4,1.9,3.51,0.0
1,9.8,2.6,3.20,0.0
2,9.8,2.3,3.26,0.0
3,9.8,1.9,3.16,0.0
4,9.4,1.9,3.51,0.0


In [3]:
# Feature와 Target
data = wine.iloc[:,:3]
target = wine.iloc[:,-1]

In [4]:
from sklearn.model_selection import train_test_split

train_input, test_input, train_target, test_target = \
  train_test_split(
    data,
    target,
    test_size=0.2,
    random_state=42
  )

### 랜덤 포레스트 모델

In [5]:
import numpy as np
from sklearn.model_selection import cross_validate
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
  n_jobs=-1,
  random_state=42
)

scores = cross_validate(
  rf,
  train_input,
  train_target,
  return_train_score=True,
  n_jobs=-1
)


print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.9973541965122431 0.8903229806766861


In [6]:
# 주요 Feature
rf.fit(train_input,train_target)
rf.feature_importances_

array([0.23183515, 0.50059756, 0.26756729])

> 결정트리보다 Sugar의 중요도가 떨어진 건 랜덤포레스트는 Feature의 갯수를 랜덤하게 사용하기 때문이다.

#### 남는 샘플(oob : out of back)을 활용

In [7]:
rf = RandomForestClassifier(
  oob_score=True,
  n_jobs=-1,
  random_state=42,
)
rf.fit(train_input, train_target)

print(rf.oob_score_)

0.8945545507023283


----
#### Extra Tree
- 기본적으로 100개의 트리
- 노드 분할시 특성의 제곱근을 갯수로 사용
- 특성의 선택을 랜덤하게 선택한다.
- 특성의 선택을 랜덤하게 하므로 속도는 랜덤 포레스트보다 빠르다.

In [8]:
from sklearn.ensemble import ExtraTreesClassifier

et = ExtraTreesClassifier(n_jobs=-1, random_state=42)

scores = cross_validate(
  et,
  train_input,
  train_target,
  return_train_score=True,
  n_jobs=-1
)


print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.9974503966084433 0.8887848893166506


----
#### Gradient Boosting
- 가장 유명한 알고리즘 중의 하나이다.
- 경사하강법처럼 손실함수를 사용
- 손실함수를 보고 트리를 추가하여 최적의 값을 도출하는 방법
- Decision Tree Regressor를 사용하여 손실함수를 계산하고 이를 계속 낮추기 위해 트리를 추가하는 구조.
- 경사를 이동하면서 경사의 이동거리를 제어하는 learning-rate(기본:0.1)를 사용.
- max-depth를 3으로 제어하여 깊이가 깊어지는 과대적합 방지
- 단점은 손실함수를 보고 트리를 추가하면서 진행하는 모델이므로 n_jobs(병렬처리)를 사용할 수 없다.

In [18]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(random_state=42)

scores = cross_validate(
  gb,
  train_input,
  train_target,
  return_train_score=True,
  n_jobs=-1
)


print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.8881086892152563 0.8720430147331015


----
#### 히스토그램 기반 그래디언트 부스팅
- 훈련데이터 256개의 구간으로 나누어서 훈련한다.
- 특성의 범위가 제한되어 빠른 속도를 제공한다.
- 제한된 구간이므로 과대적합을 방지한다.
- 아직은 실험단계인 모델이다.

In [16]:
from sklearn.ensemble import HistGradientBoostingClassifier

hgb = HistGradientBoostingClassifier(random_state=42)

scores = cross_validate(
  hgb,
  train_input,
  train_target,
  return_train_score=True,
  n_jobs=-1
)


print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.9321723946453317 0.8801241948619236


---
#### XGBoost
: Kaggle에서 많이 사용

In [11]:
# !pip install xgboost

In [29]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
  tree_method = 'hist',
  random_stsaste = 42,
  # use_label_encoder = False,
  eval_metric = 'logloss'
)

scores = cross_validate(
  xgb,
  train_input,
  train_target,
  return_train_score=True,
  n_jobs=-1
)


print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.9567059184812372 0.8783915747390243


/opt/anaconda3/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [17:32:35] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "random_stsaste" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/opt/anaconda3/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [17:32:35] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "random_stsaste" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/opt/anaconda3/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [17:32:35] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "random_stsaste" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/opt/anaconda3/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [17:32:35] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "random_stsaste" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


----
#### 번외 기능
##### Permutation Importance(치환 중요도)
- 각 특성별 Sample을 섞어서 계산을 한 후에 원래의 Sample들과의 차이를 계산하여 차이가 많이 나는 특성이 중요하다는 판단을 함.
- 어떤 모델에도 사용가능하며 특성을 파악하는 주요 기준으로 사용된다. 권장 사항
- DataSet을 구성할 때 필수적으로 사용되는 모델이며 수집한 데이터의 중요도 파악에 많이 사용되며 영양이 없는 특성은 속도와 정확도만 축내는 꼴이다.   

In [31]:
from sklearn.inspection import permutation_importance
hgb.fit(train_input, train_target)
result = permutation_importance(
  hgb,
  train_input,
  train_target,
  n_repeats=10,
  random_state=42,
  n_jobs=-1
)

print(result.importances_mean)

[0.08876275 0.23438522 0.08027708]


----

### Iris로 중요도 체크하기

In [32]:
iris = pd.read_csv("../Data/iris.csv")
iris.head()

,SepalLength,SepalWidth,PetalLength,PetalWidth,Name
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [39]:
# Feature와 Target
data = iris.iloc[:,:4]
target = iris.iloc[:,-1]

In [43]:
data

,SepalLength,SepalWidth,PetalLength,PetalWidth
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2
...,...,...,...,...
145,6.7,3.0,5.2,2.3
146,6.3,2.5,5.0,1.9
147,6.5,3.0,5.2,2.0
148,6.2,3.4,5.4,2.3


In [41]:
from sklearn.model_selection import train_test_split

train_input, test_input, train_target, test_target = \
  train_test_split(
    data,
    target,
    test_size=0.2,
    random_state=42
  )

In [42]:
from sklearn.inspection import permutation_importance
hgb.fit(train_input, train_target)
result = permutation_importance(
  hgb,
  train_input,
  train_target,
  n_repeats=10,
  random_state=42,
  n_jobs=-1
)

print(result.importances_mean)

[0.02       0.0375     0.59416667 0.14583333]
